In [1]:
import io, os, re, urllib3, requests, datetime, glob, json
import pandas as pd
import numpy as np
from datetime import date
from bs4 import BeautifulSoup
from dateutil.parser import parse
import xml.etree.ElementTree as ET
from io import BytesIO

#warnings.simplefilter("ignore", category=pd.errors.SettingWithCopyWarning)
import warnings
warnings.filterwarnings('ignore')

d = os.getcwd()+"/"
PATH_RAW = os.path.join(d, "..", "raw")   # raw inputs (downloads / intermediate CSVs)
PATH_DATA = os.path.join(d, "..", "data")    # processed modeling tables
# Create folder if it does not exist
os.makedirs(PATH_RAW, exist_ok=True)
os.makedirs(PATH_DATA, exist_ok=True)

hdr = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537'}


c:\Users\guerr\anaconda3\envs\nwcst\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


# GDP

From https://statinja.gov.jm/NationalAccounting/Quarterly/NewQuarterlyGDP.aspx


## Example

Let's check the URL

In [2]:
url = "https://www.datazoa.com/data/table.asp?a=view&th=69E287AF0E&dzuuid=1835&uid=dzadmin"

We can use pandas to open this url into a table

In [3]:
tables = pd.read_html(url)
dfi = tables[0]
dfi

,Unnamed: 0,2024 Q3,2024 Q4,2025 Q1,2025 Q2,2025 Q3,2025 Q4
0,Total Value Added at Basic Prices,424159,431334,436021,437133,438665,406765
1,Agriculture Forestry & Fishing,30645,33038,33882,34027,33460,30617
2,Mining & Quarrying,5095,5395,5882,5350,5158,3658
3,Manufacturing,39886,40104,40745,41792,42947,37010
4,Food Beverages & Tobacco,25258,25655,25659,26309,26802,23202
5,Other Manufacturing,14628,14449,15086,15483,16145,13807
6,Electricity Water Supply & Waste Management,11473,11967,12063,11835,12207,10573
7,Construction,27298,27197,27764,27988,28140,26931
8,Wholesale & Retail Trade; Repair of Motor Vehi...,70958,72442,72383,72640,72178,71214
9,Accommodation & Food Service Activities,24348,25712,25362,25874,26115,17776


Now we need to cleam this data.

First, we can change variable names

In [4]:
# Rename first column as variable name.
dfi.rename(columns={"Unnamed: 0": "variable"}, inplace=True)
dfi.head(5)

,variable,2024 Q3,2024 Q4,2025 Q1,2025 Q2,2025 Q3,2025 Q4
0,Total Value Added at Basic Prices,424159,431334,436021,437133,438665,406765
1,Agriculture Forestry & Fishing,30645,33038,33882,34027,33460,30617
2,Mining & Quarrying,5095,5395,5882,5350,5158,3658
3,Manufacturing,39886,40104,40745,41792,42947,37010
4,Food Beverages & Tobacco,25258,25655,25659,26309,26802,23202


Good coding should avoid special symbols. Simplify variable names

In [5]:
# In Column variable: eliminate special symbols, extra spaces, use lowercase, and substitute spaces with underscores.
dfi["variable"] = dfi["variable"].str.replace(r"[^a-zA-Z0-9\s]", "", regex=True).str.lower().str.replace("  "," ").str.strip().str.replace(r"\s+", "_", regex=True)
dfi

,variable,2024 Q3,2024 Q4,2025 Q1,2025 Q2,2025 Q3,2025 Q4
0,total_value_added_at_basic_prices,424159,431334,436021,437133,438665,406765
1,agriculture_forestry_fishing,30645,33038,33882,34027,33460,30617
2,mining_quarrying,5095,5395,5882,5350,5158,3658
3,manufacturing,39886,40104,40745,41792,42947,37010
4,food_beverages_tobacco,25258,25655,25659,26309,26802,23202
5,other_manufacturing,14628,14449,15086,15483,16145,13807
6,electricity_water_supply_waste_management,11473,11967,12063,11835,12207,10573
7,construction,27298,27197,27764,27988,28140,26931
8,wholesale_retail_trade_repair_of_motor_vehicle...,70958,72442,72383,72640,72178,71214
9,accommodation_food_service_activities,24348,25712,25362,25874,26115,17776


The famous reshape method

In [6]:
# Reshape dataframe from wide to long format.
dfi = pd.melt(dfi, id_vars=["variable"], var_name="date", value_name="value")
dfi.head(5)

,variable,date,value
0,total_value_added_at_basic_prices,2024 Q3,424159
1,agriculture_forestry_fishing,2024 Q3,30645
2,mining_quarrying,2024 Q3,5095
3,manufacturing,2024 Q3,39886
4,food_beverages_tobacco,2024 Q3,25258


In [7]:
# Transform value in numeric, make nan those values that cannot be converted.
dfi["value"] = pd.to_numeric(dfi["value"], errors="coerce")
dfi.head(5)

,variable,date,value
0,total_value_added_at_basic_prices,2024 Q3,424159.0
1,agriculture_forestry_fishing,2024 Q3,30645.0
2,mining_quarrying,2024 Q3,5095.0
3,manufacturing,2024 Q3,39886.0
4,food_beverages_tobacco,2024 Q3,25258.0


Drop rows that don't have a value. And transform dates

In [8]:
dfi.dropna(subset=["value"], inplace=True)

# Transform date into datetime format. Date has format 'Q1 2020' but we want it to be the first day of the first month in the quarter
dfi["date"] = dfi["date"].str.replace("Q1", "01").str.replace("Q2", "04").str.replace("Q3", "07").str.replace("Q4", "10")
dfi["date"] = pd.to_datetime(dfi["date"], format="%Y %m")

dfi.head(5)

,variable,date,value
0,total_value_added_at_basic_prices,2024-07-01,424159.0
1,agriculture_forestry_fishing,2024-07-01,30645.0
2,mining_quarrying,2024-07-01,5095.0
3,manufacturing,2024-07-01,39886.0
4,food_beverages_tobacco,2024-07-01,25258.0


Reshape but 'pivot'

In [9]:
# Pivot data to have variables as columns.
dfi = dfi.pivot(index="date", columns="variable", values="value")
dfi.head(5)

variable,accommodation_food_service_activities,agriculture_forestry_fishing,construction,education_health_other_services,electricity_water_supply_waste_management,financial_insurance_activities,food_beverages_tobacco,information_communication,manufacturing,mining_quarrying,other_manufacturing,public_administration_defence,real_estate_business_activities,total_value_added_at_basic_prices,transport_storage,wholesale_retail_trade_repair_of_motor_vehicles_installation_of_machinery_equipment
date,,,,,,,,,,,,,,,,
2024-07-01,24348.0,30645.0,27298.0,45626.0,11473.0,45109.0,25258.0,18398.0,39886.0,5095.0,14628.0,27305.0,53855.0,424159.0,24161.0,70958.0
2024-10-01,25712.0,33038.0,27197.0,46026.0,11967.0,45302.0,25655.0,18472.0,40104.0,5395.0,14449.0,27353.0,53793.0,431334.0,24533.0,72442.0
2025-01-01,25362.0,33882.0,27764.0,46837.0,12063.0,46049.0,25659.0,18522.0,40745.0,5882.0,15086.0,27521.0,54132.0,436021.0,24879.0,72383.0
2025-04-01,25874.0,34027.0,27988.0,45855.0,11835.0,46881.0,26309.0,18129.0,41792.0,5350.0,15483.0,27612.0,54025.0,437133.0,25125.0,72640.0
2025-07-01,26115.0,33460.0,28140.0,45408.0,12207.0,47242.0,26802.0,18630.0,42947.0,5158.0,16145.0,27215.0,54155.0,438665.0,25808.0,72178.0


In [10]:
dfi.rename(columns={'total_value_added_at_basic_prices':'gdp',
                    'totalvalueaddedatbasicprices':'gdp'}, inplace=True)
dfi

variable,accommodation_food_service_activities,agriculture_forestry_fishing,construction,education_health_other_services,electricity_water_supply_waste_management,financial_insurance_activities,food_beverages_tobacco,information_communication,manufacturing,mining_quarrying,other_manufacturing,public_administration_defence,real_estate_business_activities,gdp,transport_storage,wholesale_retail_trade_repair_of_motor_vehicles_installation_of_machinery_equipment
date,,,,,,,,,,,,,,,,
2024-07-01,24348.0,30645.0,27298.0,45626.0,11473.0,45109.0,25258.0,18398.0,39886.0,5095.0,14628.0,27305.0,53855.0,424159.0,24161.0,70958.0
2024-10-01,25712.0,33038.0,27197.0,46026.0,11967.0,45302.0,25655.0,18472.0,40104.0,5395.0,14449.0,27353.0,53793.0,431334.0,24533.0,72442.0
2025-01-01,25362.0,33882.0,27764.0,46837.0,12063.0,46049.0,25659.0,18522.0,40745.0,5882.0,15086.0,27521.0,54132.0,436021.0,24879.0,72383.0
2025-04-01,25874.0,34027.0,27988.0,45855.0,11835.0,46881.0,26309.0,18129.0,41792.0,5350.0,15483.0,27612.0,54025.0,437133.0,25125.0,72640.0
2025-07-01,26115.0,33460.0,28140.0,45408.0,12207.0,47242.0,26802.0,18630.0,42947.0,5158.0,16145.0,27215.0,54155.0,438665.0,25808.0,72178.0
2025-10-01,17776.0,30617.0,26931.0,43822.0,10573.0,46364.0,23202.0,16162.0,37010.0,3658.0,13807.0,27891.0,52312.0,406765.0,22436.0,71214.0


In [11]:
dfi[[ 'gdp']]

variable,gdp
date,
2024-07-01,424159.0
2024-10-01,431334.0
2025-01-01,436021.0
2025-04-01,437133.0
2025-07-01,438665.0
2025-10-01,406765.0


## Full code

In [ ]:
try :
    df = pd.read_csv(f"{folder_spec}/boj_gdp.csv", parse_dates=['date'], index_col='date')
except :
    df = pd.read_html(f"{PATH_RAW}/historic.xls")[0]

    # Read second row as header
    df.columns = df.iloc[1]
    df = df.rename(columns={df.columns[0]: "variable"})

    # In Column variable: eliminate special symbols, extra spaces, use lowercase, and substitute spaces with underscores.
    df["variable"] = df["variable"].str.replace(r"[^a-zA-Z0-9\s]", "", regex=True).str.lower().str.replace("  "," ").str.strip().str.replace(r"\s+", "_", regex=True)

    # Reshape dataframe from wide to long format.
    df = pd.melt(df, id_vars=["variable"], var_name="date", value_name="value")
    # Transform value in numeric, make nan those values that cannot be converted.
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    df.dropna(subset=["value"], inplace=True)


    # Transform date into datetime format. Date has format 'Q1 2020' but we want it to be the first day of the first month in the quarter
    df["date"] = df["date"].str.replace("Q1", "01").str.replace("Q2", "04").str.replace("Q3", "07").str.replace("Q4", "10")
    df["date"] = pd.to_datetime(df["date"], format="%Y %m")

    # Pivot data to have variables as columns.
    df = df.pivot(index="date", columns="variable", values="value")

    df.rename(columns={'total_value_added_at_basic_prices':'gdp'}, inplace=True)
    df.insert(0, "gdp", df.pop("gdp"))
    df.to_csv(f"{PATH_RAW}/boj_gdp.csv")

print(f"Latest data: {df.index.max()}") 

In [ ]:
# url = "https://statinja.gov.jm/NationalAccounting/Quarterly/NewQuarterlyGDP.aspx"
url = "https://www.datazoa.com/data/table.asp?a=view&th=69E287AF0E&dzuuid=1835&uid=dzadmin"

tables = pd.read_html(url)
dfi = tables[0]
# Rename first column as variable name.
dfi.rename(columns={"Unnamed: 0": "variable"}, inplace=True)

# In Column variable: eliminate special symbols, extra spaces, use lowercase, and substitute spaces with underscores.
dfi["variable"] = dfi["variable"].str.replace(r"[^a-zA-Z0-9\s]", "", regex=True).str.lower().str.replace("  "," ").str.strip().str.replace(r"\s+", "_", regex=True)

# Reshape dataframe from wide to long format.
dfi = pd.melt(dfi, id_vars=["variable"], var_name="date", value_name="value")
# Transform value in numeric, make nan those values that cannot be converted.
dfi["value"] = pd.to_numeric(dfi["value"], errors="coerce")

dfi.dropna(subset=["value"], inplace=True)

# Transform date into datetime format. Date has format 'Q1 2020' but we want it to be the first day of the first month in the quarter
dfi["date"] = dfi["date"].str.replace("Q1", "01").str.replace("Q2", "04").str.replace("Q3", "07").str.replace("Q4", "10")
dfi["date"] = pd.to_datetime(dfi["date"], format="%Y %m")

# Pivot data to have variables as columns.
dfi = dfi.pivot(index="date", columns="variable", values="value")
dfi.rename(columns={'total_value_added_at_basic_prices':'gdp'}, inplace=True)

# Combine dataframes

df.index  = pd.to_datetime(df.index)
dfi.index = pd.to_datetime(dfi.index)
df = pd.concat([df, dfi[dfi.index > df.index.max()]])


df = df.apply(pd.to_numeric, errors="coerce")

# df = dfi.combine_first(df)
df.to_csv(f"{PATH_RAW}/boj_gdp.csv")
print(f"Updated data: {df.index.max()}") 

## CPI

In [ ]:
url = "https://wups.statinja.gov.jm/wup/egddsfiles/CPI_Jamaica.xml"

# 1. Download XML
response = requests.get(url)
response.raise_for_status()

# 2. Parse XML from bytes
tree = ET.parse(BytesIO(response.content))
root = tree.getroot()

data = []

for series in root.findall(".//Series"):
    if series.attrib.get("INDICATOR") == "PCPI_IX":
        for obs in series.findall("Obs"):
            date = obs.attrib.get("TIME_PERIOD")
            value = obs.attrib.get("OBS_VALUE")
            if value is not None:
                data.append((date, float(value)))

df = pd.DataFrame(data, columns=["date", "cpi"])
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

df = df.apply(pd.to_numeric, errors="coerce")

df.to_csv(f"{PATH_RAW}/boj_cpi.csv")
print(f"Updated data: {df.index.max()}") 

## Labor Market

In [ ]:
url = "https://boj.org.jm/wp-content/uploads/2024/10/LMI_Jamaica.xml"
r = requests.get(url)
r.raise_for_status()

# Parse XML
root = ET.fromstring(r.content)

# Container for long-format data
rows = []

# Loop through Series
for series in root.findall(".//{*}Series"):
    indicator = series.attrib.get("INDICATOR")

    for obs in series.findall("{*}Obs"):
        rows.append({
            "date": obs.attrib.get("TIME_PERIOD"),
            indicator: float(obs.attrib.get("OBS_VALUE"))
        })

# Convert to DataFrame
df = pd.DataFrame(rows)

# Pivot to wide format (one column per indicator)
df = (
    df
    .groupby("date")
    .first()
    .sort_index()
)
idx = df.index.to_series()
year = idx.str.slice(0, 4).astype(int)
quarter = idx.str.slice(-1).astype(int)
month = quarter.map({1: 3, 2: 6, 3: 9, 4: 12})
df.index = pd.to_datetime(
    dict(year=year, month=month, day=1)
)
df.index.name = "date"

df = df.apply(pd.to_numeric, errors="coerce")

df.to_csv(f"{PATH_RAW}/boj_labor.csv")
print(f"Updated data: {df.index.max()}")

## Stock Market

In [ ]:
url = f"https://www.jamstockex.com/trading/indices/index-history/?indexCode=7&fromDate=2021-01-12"
r = requests.get(url, timeout=60, verify=False, headers=hdr)

df = pd.read_html(r.text)
df = df[0]

df.columns = df.columns.astype(str).str.lower()
df = df[['date', 'value', 'volume traded']].set_index('date')
df.rename(columns={'value':'jse_index', 'volume traded':'jse_volume'}, inplace=True)

df.index = pd.to_datetime(df.index)
df = df.sort_index()

df = df.apply(pd.to_numeric, errors="coerce")

df.to_csv(f"{PATH_RAW}/jamstockex_jse.csv")
print(f"Updated data: {df.index.max()}")

# Trade: Partner Data

### USA Trade

In [ ]:
'''
Access the EIA and download the xls file that has all data.
'''
url = "https://www.eia.gov/dnav/pet/hist/LeafHandler.ashx?n=PET&s=RBRTE&f=D"

r = requests.get(url=url, verify=None).content
soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
link = [ link for link in links if "xls" in link ][0].replace("../","")
get = f"https://www.eia.gov/dnav/pet/{link}"
df = pd.ExcelFile(get)

df = pd.read_excel( df , sheet_name=[ sheet for sheet in df.sheet_names if "data" in sheet.lower() ][0], skiprows=2)
df.columns = df.columns.str.lower()
df.set_index('date', inplace=True)
df = df.rename(columns=lambda x: "brent price" if "brent" in x else x)
df.index = pd.to_datetime(df.index)

df.to_csv(f"{PATH_RAW}/eia_brent.csv")
print(f"EIA  Updated: {df.index.max()}")

'''
Access the US Census and dowload the xls file with all the data
'''
url = "https://www.census.gov/foreign-trade/balance/c2360.html"

r = requests.get(url=url, verify=None).content
soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
get = [ link for link in links if "xls" in link ]
get = f"https://www.census.gov/{get[0]}"

#df = pd.read_excel(get)
df = pd.read_excel(BytesIO(requests.get(get, headers=hdr).content))
df = df[df['CTYNAME'].str.contains("Jamaica")]

df = df.drop(['CTY_CODE', 'CTYNAME', 'IYR', 'EYR'], axis=1)
df.columns = df.columns.str.lower()
df = pd.melt(df, id_vars="year", var_name="month", value_name='trade')

df['variable'] = df['month'].str[0]
df['month'] = df['month'].str[1:]
df['variable'] = df['variable'].replace({'i': 'USA_imp_JM', 'e': 'USA_exp_JM'})
df.columns = df.columns.astype(str).str.strip()

df['date'] = pd.to_datetime(df['year'].astype(str)+df['month'], format="%Y%b") + pd.offsets.MonthEnd(0)
df = df.set_index('date')[['trade', 'variable']]

df = df.pivot(columns='variable', values='trade')
df = df[df["USA_exp_JM"] != 0]

df.to_csv(f"{PATH_RAW}/uscensus_trade.csv")
print(f"US Census Updated: {df.index.max()}")

### Canada Trade

In [ ]:
import zipfile
import io
import shutil
def extract_year(url):
    match = re.search(r'(20\d{2})', url)
    if match:
        return int(match.group(1))
    return None

try :
    df = pd.read_csv(f"{PATH_RAW}/canada_imports.csv", index_col='date', parse_dates=True)
    #2022-2025 Files
    url = "https://open.canada.ca/data/en/dataset/2909a648-5753-4924-878a-b069392d9cde"
except :
    df = pd.DataFrame()
    url = [
        "https://open.canada.ca/data/en/dataset/2909a648-5753-4924-878a-b069392d9cde",
        "https://open.canada.ca/data/en/dataset/b1126a07-fd85-4d56-8395-143aba1747a4", 
        "https://open.canada.ca/data/en/dataset/6f3fd3d9-5a35-47a8-a45a-4b2a8c957453",
        "https://open.canada.ca/data/en/dataset/d0c43a70-3390-4879-8f1d-b602bb0eba3d",
        ]

links = []
if isinstance(url, list) :
    for u in url :
        r = requests.get(url=u, verify=None).content
        soup = BeautifulSoup(r)
        l = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
        l = [ link for link in l if "Imp" in link ]
        links = links + l
else :
    r = requests.get(url=url, verify=None).content
    soup = BeautifulSoup(r)
    links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
    links = [ link for link in links if "Imp" in link ]
    links = [ link for link in links if extract_year(link.split("/")[-1]) >= df.index.max().year ]


In [ ]:
for link in links : 
    # Download the Zip file
    response = requests.get(link)
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        z.extractall(f"{PATH_RAW}")  # Extract to a folder
        print("ZIP extracted successfully!")

    # Extract file 022 (imports by product)
    file_name = [ x for x in z.namelist() if "N022" in x ][0]
    #file_name = [ x for x in z.namelist() if "N022" in x.split("/")[-1] ][0]
    file_name = f"{PATH_RAW}/{file_name}"
    # Open the file
    dfi = pd.read_csv(file_name)

    # Clean variable names
    for col in dfi.columns :
        if "YearMonth" in col :
            dfi = dfi.rename(columns = {col : 'date'} )
        elif "Country" in col :
            dfi = dfi.rename(columns = {col : 'partner'} )
        elif "Value" in col :
            dfi = dfi.rename(columns = {col : 'value'} )
        elif "Quantity" in col :
            dfi = dfi.rename(columns = {col : 'quantity'} )
        elif "Unit" in col :
            dfi = dfi.rename(columns = {col : 'unit'} )

    # Find Bahamas
    dfi = dfi[dfi['partner']=='JM']
    # Clean date
    dfi['date'] = pd.to_datetime(pd.to_datetime(dfi['date'] , format="%Y%m").dt.strftime('%Y-%m-%d'))
    dfi = dfi.set_index('date')

    # Sum all monthly values
    dfi = dfi.resample('MS')[['value']].sum()
    dfi.rename(columns = {"value" : 'imports_can'} , inplace=True)
    df = df[~df.index.isin(dfi.index.unique())]
    df = pd.concat([df, dfi], axis=0)

    try :
        #folder_path = os.path.dirname(os.path.abspath(file_name))
        os.remove(file_name)
    except :
        continue

df = df.sort_index()
print(f"Updated: {df.index.max()}")
df.to_csv(f'{PATH_RAW}/canada_imports.csv')

In [ ]:
# EXPORTS
try :
    df = pd.read_csv(f"{PATH_RAW}/canada_exports.csv", index_col='date', parse_dates=True)
    #2022-2025 Files
    url = "https://open.canada.ca/data/en/dataset/2909a648-5753-4924-878a-b069392d9cde"
except :
    df = pd.DataFrame()
    url = [
        "https://open.canada.ca/data/en/dataset/2909a648-5753-4924-878a-b069392d9cde",
        "https://open.canada.ca/data/en/dataset/b1126a07-fd85-4d56-8395-143aba1747a4", 
        "https://open.canada.ca/data/en/dataset/6f3fd3d9-5a35-47a8-a45a-4b2a8c957453",
        "https://open.canada.ca/data/en/dataset/d0c43a70-3390-4879-8f1d-b602bb0eba3d",
        ]

if isinstance(url, list) :
    for u in url :
        r = requests.get(url=u, verify=None).content
        soup = BeautifulSoup(r)
        l = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
        l = [ link for link in l if "tot_exp" in link.lower() ]
        links = links + l
else :
    r = requests.get(url=url, verify=None).content
    soup = BeautifulSoup(r)
    links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
    links = [ link for link in links if "tot_exp" in link.lower() ]
    links = [ link for link in links if extract_year(link.split("/")[-1]) >= df.index.max().year ]

In [ ]:
for link in links : 
    response = requests.get(link)
    
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        z.extractall(f"{PATH_RAW}")  # Extract to a folder
        print("ZIP extracted successfully!")
        
    file_name = [ x for x in z.namelist() if "N021" in x ][0]
    #file_name = [ x for x in z.namelist() if "N021" in x.split("/")[-1] ][0]
    file_name = f"{PATH_RAW}/{file_name}"
    dfi = pd.read_csv(file_name)
    for col in dfi.columns :
        if "YearMonth" in col :
            dfi = dfi.rename(columns = {col : 'date'} )
        elif "Country" in col :
            dfi = dfi.rename(columns = {col : 'partner'} )
        elif "Value" in col :
            dfi = dfi.rename(columns = {col : 'value'} )
        elif "Quantity" in col :
            dfi = dfi.rename(columns = {col : 'quantity'} )
        elif "Unit" in col :
            dfi = dfi.rename(columns = {col : 'unit'} )
    dfi = dfi[dfi['partner']=='JM']
    dfi['date'] = pd.to_datetime(pd.to_datetime(dfi['date'] , format="%Y%m").dt.strftime('%Y-%m-%d'))
    dfi = dfi.set_index('date')
    dfi = dfi.resample('MS')[['value']].sum()
       
    dfi.rename(columns = {"value" : 'exports_can'} , inplace=True)
    df = df[~df.index.isin(dfi.index.unique())]
    df = pd.concat([df, dfi], axis=0)

    try :
        #folder_path = os.path.dirname(os.path.abspath(file_name))
        os.remove(file_name)
        #if os.path.exists(folder_path): shutil.rmtree(folder_path)
    except :
        continue

df = df.sort_index()
print(f"Updated: {df.index.max()}")
df.to_csv(f'{PATH_RAW}/canada_exports.csv')

### EU Trade

In [ ]:
'''
Download data from eurostat.
eurostat package is required.
'''
# conda install conda-forge::eurostat
import eurostat
dataset = 'ds-045409'
params = {
    'format': 'TSV',        # Use uncompressed TSV format for simplicity
    'geo': 'EU',            # Specify the region (EU)
    'partner': 'JM',        # Specify the partner (Bahamas)
    'time': '2000',         # You can specify a smaller time range (e.g., 2021)
    'trade': '1',           # You can filter for imports (1) or exports (2)
    'product': '0',         # You can specify product codes if needed
    'compressed': 'false',  # Disable compression
}
df = eurostat.get_data_df(dataset,
                       filter_pars = {
                           'reporter' : 'EU' ,
                           'partner' : 'BS' ,
                           'indicators' : 'VALUE_IN_EUROS',
                           'freq' : 'M' ,
                           'product' : 'TOTAL'
                       })
df.drop(['freq', 'reporter', 'partner', 'product'], axis=1, inplace=True)

df = pd.melt(df,
        id_vars=['flow', 'indicators\\TIME_PERIOD'],
        var_name='date', value_name='value')

df['date'] = pd.to_datetime(pd.to_datetime(df['date'] , format="%Y-%m").dt.strftime('%Y-%m-%d'))
df['flow'] = df['flow'].map({"1": 'imp_eu', "2": 'exp_eu'})
df.set_index('date', inplace=True)
df = df.pivot_table(index=df.index, columns='flow', values='value')

print(f"Updated: {df.index.max()}")
df.to_csv(f'{PATH_RAW}/eu_trade.csv')

### UK Trade

In [ ]:
# Prepare all functions to read data.

import pprint as pp
import logging

logger = logging.getLogger(__name__)

ROOT_URL = "https://api.beta.ons.gov.uk/v1/"


def get_list_of_datasets():
    """Get list of all datasets available from API.
    Currently (August 2021), there are 41 available datasets.

    Returns
    -------
    list of dicts
        Metadata objects for each available dataset.
    """
    num_to_get = 100
    datasets = []

    offset = 0
    while len(datasets) < num_to_get:
        r = requests.get(ROOT_URL + "datasets", params={"offset": offset})
        results = r.json()
        [logger.info(item.get("title")) for item in results.get("items")]
        datasets.extend(results.get("items"))
        num_retrieved = results.get("count")
        offset += num_retrieved
        if num_retrieved == 0:
            break
    logger.info(f"\nFound {len(datasets)} datasets")
    return datasets


def get_dataset_by_name(datasets, target_name):
    """Get a dataset by matching on its title (or part of the title).
    Returns the first dataset object whose name contains the given target_name string.

    Parameters
    ----------
    datasets : List
        List of dataset objects
    target_name : str
        name (or partial name) of target dataset

    Returns
    -------
    dict
        Dataset object (or None, if no match is found)
    """
    for ds in datasets:
        if target_name.lower() in ds.get("title").lower():
            logger.info(f"Found dataset '{ds.get('title')}'")
            return ds
    logger.info(f"No dataset found containing '{target_name}'")
    return None


def get_edition(dataset, prefered_edition="time-series"):
    """Get one edition of a dataset. If no preferred edition is
    specified, return the most recent one.

    Parameters
    ----------
    dataset : dict
        dataset metadata
    prefered_edition : str, optional
        name of edition, by default "time-series"

    Returns
    -------
    str
        URL of edition
    """
    editions_url = dataset.get("links").get("editions").get("href")
    r = requests.get(editions_url)
    results = r.json()
    for row in results.get("items"):
        if row.get("edition") == prefered_edition:
            edition = row.get("links").get("latest_version").get("href")
            return edition

    # Default to latest version, if requested version is not found.
    latest_version = dataset.get("links").get("latest_version").get("href")
    return latest_version


def get_dimensions(edition_url):
    """Builds dictionary of all valid options for all dimensions of a given dataset,
    with descriptions for each option.
    Individual obvserviations can later be obtained by choosing from these options.
    Ranges can later be obtained by replacing one dimesion with the wildcard '*'.

    Parameters
    ----------
    dataset : dict
        single dataset

    Returns
    -------
    dict of dicts
        map of {dimensions:{acceptable_values:description}}
    """
    valid_dimensions = {}
    r = requests.get(edition_url + "/dimensions")
    results = r.json()
    for dimension in results.get("items"):
        logger.info(f'{dimension.get("name")}: \t{dimension.get("label")}')
        dim_id = dimension.get("links").get("options").get("id")
        options_url = f"{edition_url}/dimensions/{dim_id}/options"

        sr = requests.get(options_url, params={"limit": 50})
        sresults = sr.json()
        # TODO! Could add in paging here, as there *could* be multiple pages of valid options.
        logger.info(f"\tHas {sresults.get('count')} options")
        # valid_options = [item.get("option") for item in sresults.get("items")]
        option_descriptions = {
            item.get("option"): item.get("label") for item in sresults.get("items")
        }
        logger.info(f'{dimension.get("name")}: {option_descriptions}')
        valid_dimensions[dimension.get("name")] = option_descriptions

    return valid_dimensions


def choose_dimensions(valid_dims, overrides={}):
    """For each dimension, choose a single valid option (except for 'time', where
    we use the wildcard '*' to get the whole time-series.)
    If not specified, choose the first valid option for each dimension.

    Parameters
    ----------
    valid_dims : dict
        map of lists of valid dimension values
    overrides : dict, optional
        selected dimensions

    Returns
    -------
    dict
        final choice of dimensions
    """
    # By default, choose first valid item for all dimensions; then override where needed:
    chosen_dimensions = {k: next(iter(v.keys())) for k, v in valid_dims.items()}
    # get whole time-series, not just a single point in time:
    chosen_dimensions["time"] = "*"
    chosen_dimensions.update(overrides)
    return chosen_dimensions


def get_observations(edition_url, dimensions):
    """[summary]

    Parameters
    ----------
    edition_url : str
        URL of this edition of the data
    dimensions : dict
        dimensions specifying slice of data required

    Returns
    -------
    pd.Dataframe
        Summary of data, with columns "id" (time) and "observation" (value)
    """
    r = requests.get(edition_url + "/observations", params=dimensions)
    results = r.json()
    summary = []
    for observation in results.get("observations"):
        id = observation.get("dimensions").get("Time").get("id")
        summary.append({"id": id, "observation": observation.get("observation")})
    df = pd.DataFrame(summary)
    return df


def get_timeseries(dataset_name, dimension_values):
    """Get a specified dataset time-series, with a given set of dimensions.
    NB: dataframe is not sorted.

    Parameters
    ----------
    dataset_name : str
        Descriptive name of data set.
    dimension_values : dict
        set of valid dimensions for this dataset.
        If set to "None", then return set of valid dimesions.

    Returns
    -------
    Either:
        dict
            Set of valid dimensions, if None specified in function call
    or:
        dataframe
            containing time series
        dict
            dataset metadata
        str
            url of this edition of the data
    """
    dss = get_list_of_datasets()
    ds = get_dataset_by_name(dss, dataset_name)
    edition_url = get_edition(ds)
    valid_dims = get_dimensions(edition_url)
    logger.info(valid_dims)
    if dimension_values is None:
        return valid_dims
    chosen_dimensions = choose_dimensions(valid_dims, dimension_values)
    df = get_observations(edition_url, chosen_dimensions)
    logger.info(df.shape)
    return df, ds, edition_url


def demo():
    print("=" * 70)
    print("List of available datasets:")
    dss = get_list_of_datasets()
    [print(item.get("title")) for item in dss]

    print("=" * 70)
    dataset_name = "UK Labour Market"
    print(f"Valid options for dimensions for the {dataset_name}, with descriptions")
    # Get the set of valid dimensions for the Labour Market set.
    # E.g. list of valid age groups, economic activity categories etc.

    dimensions = get_timeseries(dataset_name, None)
    pp.pprint(dimensions)
    print("\n")

    # We've now selected specific dimensions for our request:
    labour_market_dimensions = {
        "economicactivity": "in-employment",
        "agegroups": "16+",
        "seasonaladjustment": "seasonal-adjustment",
        "sex": "all-adults",
        "unitofmeasure": "rates",
    }
    print(f"Chosen dimensions for the {dataset_name}")
    pp.pprint(labour_market_dimensions, indent=4)

    df_labour = get_timeseries(dataset_name, labour_market_dimensions)[0]
    df_labour["year"] = (
        df_labour["id"].str[-4:].astype(int)
    )  # extract year as last 4 digits of row id
    df_labour = df_labour.sort_values("year")
    print("")
    print(df_labour)
    print("\n")

    # Repeat the process for a second time series, GDP, with specific dimensions:
    print("=" * 70)
    gdp_dataset_name = "annual GDP"
    gdp_dimensions = {
        "geography": "UK0",
        "unofficialstandardindustrialclassification": "A--T",
    }
    print(f"Chosen dimensions for the {dataset_name}")
    df_gdp = get_timeseries(gdp_dataset_name, gdp_dimensions)[0]
    df_gdp = df_gdp.sort_values("id")
    pp.pprint(gdp_dimensions, indent=4)
    print("")
    print(gdp_dataset_name)
    print(df_gdp)

    print("=" * 70)
    print("End of demo!")

In [ ]:
#dss = get_list_of_datasets()
dataset_name = "Trade in goods: country by commodity"
#dss = [ item for item in dss if item.get("title")==dataset_name ]
#dimensions = get_timeseries(dataset_name, None)
#dimensions

df = pd.DataFrame()

for var in ['exp_uk' , 'imp_uk'] :
    if var == "exp_uk" : t = 'EX'
    elif var == "imp_uk" : t = 'IM'
    params = {
        "countriesandterritories": "JM",
        "standardindustrialtradeclassification": "T",
        #"standardindustrialtradeclassification": "0",
        'geography': 'K02000001' ,
        'direction': t ,
        }
    dfi = get_timeseries(dataset_name, params)
    dfi = dfi[0]
    dfi['date'] = pd.to_datetime(dfi['id'], format="%b-%y")
    dfi = dfi.set_index('date')
    dfi.rename(columns = { 'observation' : var} , inplace=True)
    dfi = dfi[[var]]
    dfi[var] = pd.to_numeric(dfi[var])
    
    df = pd.concat( [df, dfi], axis=1)

print(f"Updated: {df.index.max()}")
df.to_csv(f'{PATH_RAW}/uk_trade.csv')

## FRED

In [ ]:
api_key = "72304a91a13cc0ea6c610ae5845c7493"
series = ["EXCAUS", "DEXUSEU", "DEXUSUK", "SP500", "DJIA", "NASDAQCOM"]

df = pd.DataFrame()
for serie in series :
    #&realtime_end=9999-12-31
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={serie}&api_key={api_key}&file_type=json"
    response = requests.get(url)
    response = response.json()
    dfi = pd.DataFrame(response['observations'])
    dfi = dfi[['value', 'date']]
    dfi.set_index('date', inplace=True)
    dfi.index = pd.to_datetime(dfi.index)
    dfi.rename(columns = { "value" : f"fred_{serie}" }, inplace=True)
    dfi[f"fred_{serie}"] = dfi[f"fred_{serie}"].apply(pd.to_numeric, errors='coerce')
    df = df.merge(dfi, left_index=True, right_index=True, how='outer')
    df = pd.DataFrame(df.resample("D").mean())
    #df = pd.concat([df, dfi], axis=1)
    df.sort_index(inplace=True)
    
df.to_csv(f"{PATH_RAW}/fred_data.csv")
print(f"FRED Updated: {df.index.max()}")